In [2]:
import os
from typing import Any, Dict, Optional
from mcp.server.fastmcp import FastMCP
from sqlalchemy import create_engine, text

from typing import TypedDict

class HighestUrgencyTicket(TypedDict, total=False):     # `total=False`: None of these keys are strictly required
    ticket_id: str
    user_id: str
    account_id: str
    ticket_content: str
    ticket_tag: str
    ticket_channel: str
    ticket_urgency: float

# --- Configuration ---
UDAHUB_DB_PATH = os.getenv("UDAHUB_DB_PATH", "data/core/udahub.db")
DB_URL = f"sqlite:///{UDAHUB_DB_PATH}"
engine = create_engine(DB_URL)


In [3]:
query = text("""
    SELECT 
        t.ticket_id, 
        tm.content, 
        u.external_user_id AS owner_id, 
        t.channel, 
        tmd.tags, 
        t.account_id, 
        tmd.urgency_score
    FROM tickets t
    JOIN ticket_metadata tmd ON t.ticket_id = tmd.ticket_id
    JOIN users u ON t.user_id = u.user_id
    JOIN ticket_messages tm ON t.ticket_id = tm.ticket_id
    ORDER BY tmd.urgency_score DESC
    LIMIT 1
""")

In [ ]:
def get_highest_urgency_ticket() -> HighestUrgencyTicket:
    """
    Fetches the ticket with the highest urgency score using raw SQL.
    """
    query = text("""
        SELECT 
            t.ticket_id, 
            tm.content, 
            u.external_user_id AS owner_id, 
            t.channel, 
            tmd.tags, 
            t.account_id, 
            tmd.urgency_score
        FROM tickets t
        JOIN ticket_metadata tmd ON t.ticket_id = tmd.ticket_id
        JOIN users u ON t.user_id = u.user_id
        JOIN ticket_messages tm ON t.ticket_id = tm.ticket_id
        ORDER BY tmd.urgency_score DESC
        LIMIT 1
    """)

    try:
        with engine.connect() as conn:
            result = conn.execute(query).mappings().first()
            if result is None:
                result = HighestUrgencyTicket({})
            else:
                result = dict(result)
                result["user_id"] = result.pop("owner_id")
                result["account_id"] = result.pop("account_id")
                result["ticket_id"] = result.pop("ticket_id")
                result["ticket_content"] = result.pop("content")
                result["ticket_tag"] = result.pop("tags")
                result["ticket_channel"] = result.pop("channel")
                result["ticket_urgency"] = result.pop("urgency_score")
                result = HighestUrgencyTicket(result)
    except Exception as e:
        result = HighestUrgencyTicket({})
    return result

In [8]:
get_highest_urgency_ticket()

{}